# EVEE 啟發的 SH2 讀出小試

以使用者提供的 Pearce et al. EVEE v1 全文（doi:10.64898/2026.04.10.717844）Methods pp. 7–8 及 Figure S5 核對設計。S5 的 Top-K 256 與連續 1,024 點比較是長上下文結果。本實驗採用同一組 32 點的均值與二階讀出，直接比較池化方式。

- 凍結既有的蒸餾前及蒸餾後 SH2，輸入沿用 GRCh38 256 bp REF/ALT。
- 固定正向，從零基索引 128 起連續取 32 點。計算 ALT−REF，三種讀出共用訓練集投影。
- 預先指定 16 維，對照中心、均值及對稱未中心化二階矩。二階矩對整體差分反號不變。
- L2 正則只用訓練內的基因分組交叉驗證選擇。主要比較為蒸餾編碼器的「二階減均值」驗證 AUPRC，配對基因群組 bootstrap 95% 區間下界高於零才追加 32 維。
- 驗證閾值以至少 90% specificity 為條件。鎖定全部候選和閾值後才開啟歷史測試，確認性評估需新留出基因或時間隊列。
- CPU 本地運行，Evo 2 API 請求 0，推理模型小於 2M。所有分數供研究。

在專案根目錄執行以下 PowerShell 指令。完整程式輸出寫入單獨的 executed Notebook。

初次 L-BFGS 在訓練內交叉驗證達到迭代上限。此次續跑使用 float64 標準化及 Newton–Cholesky 求解同一個 L2 邏輯回歸問題。初次日誌保存在原運行目錄。

    .\.venv\Scripts\python.exe scripts/run_hyenadna_notebook.py --notebook EVEE_readout_pilot.ipynb --output EVEE_readout_pilot.executed.ipynb --status outputs/evee_readout_pilot_20261109_run02/notebook_execution.json

In [1]:
import hashlib
import json
import os
from pathlib import Path
import sys
from datetime import datetime, timezone
from importlib.metadata import version

ROOT = Path.cwd().resolve()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
os.environ["RUN_EVO2_INTEGRATION"] = "0"
import torch
from evo2_mvp.utils import save_json, sha256_file
from hyenadna_notebook_support import run_stage

torch.set_num_threads(4)
RUN = Path("outputs/evee_readout_pilot_20261109_run02")
SOURCE = Path("outputs/sh2_two_stage_20261104")
DATA = SOURCE / "data"
ENCODERS = SOURCE / "training"
VARIANTS = Path("data/variants.csv")
CONFIG = Path("configs/stripedhyena2_two_stage.yaml")
FASTA = Path("data/GRCh38.fa")
RUN.mkdir(parents=True, exist_ok=True)

PAPER_SHA256 = "e91d2ea398365ddb92f9aa614dbd4ac69acc43297b80b368e59a509dc9d93e87"
paper = Path(r"D:\Users\60152\Downloads\2026.04.10.717844v1.full.pdf")
if paper.is_file():
    assert sha256_file(paper) == PAPER_SHA256, "Unexpected PDF version"

files = [
    "EVEE_readout_pilot.ipynb", str(VARIANTS), str(CONFIG), str(FASTA), "data/splits.json",
    str(DATA / "metadata.json"), str(DATA / "probe_train.npz"), str(DATA / "probe_val.npz"),
    str(ENCODERS / "distillation_status.json"), str(ENCODERS / "initial_encoder.pt"),
    str(ENCODERS / "best_encoder.pt"), "scripts/hyenadna_notebook_support.py",
    "scripts/run_hyenadna_notebook.py",
]
files += [p.as_posix() for p in sorted(Path("src/evo2_mvp").glob("*.py"))]
test_files = sorted(Path("tests").glob("test_evee*.py"))
assert test_files, "EVEE pilot verification tests are missing"
files += [p.as_posix() for p in test_files]
description = {
    "files_sha256": {Path(p).as_posix(): sha256_file(p) for p in files},
    "versions": {package: version(package) for package in ("torch", "numpy", "scikit-learn", "nbclient")},
    "paper_sha256": PAPER_SHA256, "seed": 42, "device": "cpu", "threads": 4,
}
IDENTITY = {"identity": hashlib.sha256(json.dumps(description, sort_keys=True).encode()).hexdigest(), **description}
save_json(RUN / "notebook_input_identity.json", IDENTITY)

metadata = json.loads((DATA / "metadata.json").read_text(encoding="utf-8"))
assert metadata["status"] == "complete" and metadata["synthetic"] is False
assert metadata["source_hashes"]["fasta_sha256"] == IDENTITY["files_sha256"]["data/GRCh38.fa"]
assert metadata["counts"]["probe_train"] == 1200 and metadata["counts"]["probe_val"] == 400
print("Input identity", IDENTITY["identity"], "train/validation", 1200, 400)

Input identity 460138445e214ae75e1ff92f8450c7b845e178f0b256b0e148040c0d299b04bd train/validation 1200 400


In [2]:
run_stage(ROOT, RUN, "tests", ["-m", "pytest", "-q", *map(str, test_files),
    "--basetemp", "work/pytest_evee_readout", "--junitxml", str(RUN / "junit.xml")],
    IDENTITY, [str(RUN / "junit.xml")])

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m pytest -q tests\test_evee_pilot_data.py tests\test_evee_readout_pilot.py --basetemp work/pytest_evee_readout --junitxml outputs\evee_readout_pilot_20261109_run02\junit.xml


...........                                                              [100%]


- generated xml file: D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\outputs\evee_readout_pilot_20261109_run02\junit.xml -


11 passed in 14.20s


{'status': 'completed',
 'signature': {'experiment_identity': '460138445e214ae75e1ff92f8450c7b845e178f0b256b0e148040c0d299b04bd',
  'arguments': ['-m',
   'pytest',
   '-q',
   'tests\\test_evee_pilot_data.py',
   'tests\\test_evee_readout_pilot.py',
   '--basetemp',
   'work/pytest_evee_readout',
   '--junitxml',
   'outputs\\evee_readout_pilot_20261109_run02\\junit.xml']},
 'started_at_utc': '2026-11-09T06:28:45.074155+00:00',
 'completed_at_utc': '2026-11-09T06:29:00.959329+00:00',
 'returncode': 0,
 'artifacts_sha256': {'outputs/evee_readout_pilot_20261109_run02/junit.xml': 'd02056343c1e3745fce70584a0d361fb8e15349dd7b8772669f792e65089e9db'}}

In [3]:
COMMON = [
    "--data-dir", str(DATA), "--encoders", str(ENCODERS), "--variants", str(VARIANTS),
    "--encoder-config", str(CONFIG), "--output", str(RUN), "--seed", "42", "--device", "cpu",
]
from evo2_mvp.evee_readout_pilot import read_locked_candidates
run_stage(ROOT, RUN, "fit", ["-m", "evo2_mvp.evee_readout_pilot", "--stage", "fit", *COMMON],
    IDENTITY, [str(RUN / name) for name in ("pilot_validation.json", "pilot_lock.json", "validation_predictions.csv")],
    validate=lambda: read_locked_candidates(RUN))
validation = json.loads((RUN / "pilot_validation.json").read_text(encoding="utf-8"))
print(json.dumps(validation, ensure_ascii=False, indent=2))

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m evo2_mvp.evee_readout_pilot --stage fit --data-dir outputs\sh2_two_stage_20261104\data --encoders outputs\sh2_two_stage_20261104\training --variants data\variants.csv --encoder-config configs\stripedhyena2_two_stage.yaml --output outputs\evee_readout_pilot_20261109_run02 --seed 42 --device cpu


{"status": "validation_complete", "primary": {"candidate": "distilled_gram_r16", "baseline": "distilled_mean_r16", "split": "validation", "delta_auprc": -0.0033026934031527566, "delta_auprc_95_ci": [-0.09587364964918411, 0.08740525482937714], "rule": "delta_auprc_positive_and_paired_group_bootstrap_95_ci_lower_bound_positive", "passed": false}, "rank32_triggered": false}


{
  "status": "validation_complete",
  "protocol": {
    "version": 1,
    "seed": 42,
    "orientation": "forward_reference",
    "window": 256,
    "token_start_zero_based": 128,
    "tokens": 32,
    "pca": "shared_across_arms_and_readouts_centered_fit_untranslated_projection",
    "gram": "upper_triangle_of_uncentered_symmetric_ZtZ_divided_by_token_count",
    "arms": [
      "random",
      "distilled"
    ],
    "readouts": [
      "center",
      "mean",
      "gram"
    ],
    "ranks": [
      16
    ],
    "inner_cv": "stratified_gene_connected_group_5fold_ap_for_logistic_L2",
    "cv_heldout_id_hashes": [
      "0c43af3091d84b69ba6b65a07188c12d345c225c4104cac2847cf169c827f004",
      "45939abbbc78743f0460cf1c665b032bc71cdc8a29b8b9d4873560891f0d510c",
      "b65a7f26f4fe8c554bfeb58e4eed581bcd0399989c78e38312a9400c90616482",
      "2d855b54221e9d7a95c73d869d1832627531fa756c3a591cfae27e80bdb39f50",
      "37a3ad9226d6a85c9c49a53ed7d4827f5c934094f72765c8b158374f4b25b367"
    ],
 

In [4]:
run_stage(ROOT, RUN, "describe_test", ["-m", "evo2_mvp.evee_readout_pilot", "--stage", "describe-test", *COMMON],
    IDENTITY, [str(RUN / name) for name in ("historical_test_description.json", "historical_test_predictions.csv")])
historical = json.loads((RUN / "historical_test_description.json").read_text(encoding="utf-8"))
assert historical["test_used_for_selection"] is False
import pandas as pd
from IPython.display import display
rows = []
for name, model in validation["models"].items():
    val = model["validation"]
    test = historical["models"][name]["test"]
    rows.append({"model": name, "features": model["feature_width"],
                 **{f"val_{k}": val[k] for k in ("auprc", "auroc", "brier_score", "recall", "specificity")},
                 **{f"historical_{k}": test[k] for k in ("auprc", "auroc", "brier_score", "recall", "specificity")}})
table = pd.DataFrame(rows)
table.to_csv(RUN / "model_comparison.csv", index=False)
display(table.round(4))

PowerShell: & "D:\Users\60152\learning materials\CISC7021\Report\evo2_pathogenicity_mvp\.venv\Scripts\python.exe" -u -m evo2_mvp.evee_readout_pilot --stage describe-test --data-dir outputs\sh2_two_stage_20261104\data --encoders outputs\sh2_two_stage_20261104\training --variants data\variants.csv --encoder-config configs\stripedhyena2_two_stage.yaml --output outputs\evee_readout_pilot_20261109_run02 --seed 42 --device cpu


{"status": "historical_test_described"}


,model,features,val_auprc,val_auroc,val_brier_score,val_recall,val_specificity,historical_auprc,historical_auroc,historical_brier_score,historical_recall,historical_specificity
0,random_center_r16,16,0.2416,0.5075,0.1522,0.1892,0.9018,0.1605,0.4921,0.1069,0.1875,0.9119
1,random_mean_r16,16,0.2112,0.4789,0.1527,0.1351,0.9049,0.1438,0.5023,0.1070,0.1250,0.9091
2,random_gram_r16,136,0.3239,0.6115,0.1568,0.2162,0.9172,0.2058,0.6369,0.1225,0.2083,0.9148
3,distilled_center_r16,16,0.2488,0.6150,0.1494,0.1486,0.9172,0.2077,0.6139,0.1046,0.2083,0.9290
4,distilled_mean_r16,16,0.3231,0.6545,0.1453,0.2162,0.9049,0.2101,0.6062,0.1048,0.2292,0.9347
5,distilled_gram_r16,136,0.3198,0.6763,0.1447,0.2432,0.9018,0.2150,0.6855,0.1057,0.2708,0.9148


In [5]:
from evo2_mvp.evee_readout_pilot import load_pilot_model
from evo2_mvp.models import one_hot
from evo2_mvp.sequence import ReferenceGenome, Variant

# Predetermined reload example, independent of validation and test ranking.
example_path = RUN / "models/distilled_gram_r16.pt"
first, _ = load_pilot_model(example_path)
second, _ = load_pilot_model(example_path)
variant = Variant("1", 926029, "C", "T")
with ReferenceGenome(FASTA) as reference:
    context = reference.context(variant, 256)
ref = one_hot([context.ref_sequence])
alt = one_hot([context.alt_sequence])
with torch.inference_mode():
    first_logits, second_logits = first(ref, alt), second(ref, alt)
assert torch.isfinite(first_logits).all() and torch.equal(first_logits, second_logits)
p = first_logits.sigmoid().item()
import math
q = min(max(p, 1e-12), 1 - 1e-12)
prediction = {"variant_id": variant.id, "genome_build": "GRCh38", "student_score": p,
              "student_predictive_entropy": -q * math.log(q) - (1-q) * math.log(1-q),
              "model": "distilled_gram_r16", "calibration_status": "not_calibrated",
              "reload_logits_identical": True, "clinical_use": "research_only"}
save_json(RUN / "standalone_prediction.json", prediction)
for name in ("initial_encoder.pt", "best_encoder.pt"):
    path = ENCODERS / name
    assert sha256_file(path) == IDENTITY["files_sha256"][path.as_posix()]
report = {"status": "completed", "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_identity": IDENTITY["identity"], "paper_sha256": PAPER_SHA256,
    "validation_report_sha256": sha256_file(RUN / "pilot_validation.json"),
    "candidate_lock_sha256": sha256_file(RUN / "pilot_lock.json"),
    "historical_test_report_sha256": sha256_file(RUN / "historical_test_description.json"),
    "standalone_prediction": prediction, "new_evo2_api_calls": 0,
    "evaluation_scope": "exploratory_validation_and_locked_historical_test",
    "clinical_use": "research_only"}
save_json(RUN / "run_report.json", report)
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "status": "completed",
  "completed_at_utc": "2026-11-09T06:30:26.023710+00:00",
  "experiment_identity": "460138445e214ae75e1ff92f8450c7b845e178f0b256b0e148040c0d299b04bd",
  "paper_sha256": "e91d2ea398365ddb92f9aa614dbd4ac69acc43297b80b368e59a509dc9d93e87",
  "validation_report_sha256": "063580130db488cf1947368b442549a61ae121e234c9cddcfbd06af0f95f576a",
  "candidate_lock_sha256": "197ececfa5f12599c5bbac05073161e65ef315336a2dd3dfde559d3109e49999",
  "historical_test_report_sha256": "dae6ccc17f0180ca93d73447bf84887965370943a3c893f93c90e1443d1fddd8",
  "standalone_prediction": {
    "variant_id": "1-926029-C-T",
    "genome_build": "GRCh38",
    "student_score": 0.11841541122857892,
    "student_predictive_entropy": 0.36375587447108165,
    "model": "distilled_gram_r16",
    "calibration_status": "not_calibrated",
    "reload_logits_identical": true,
    "clinical_use": "research_only"
  },
  "new_evo2_api_calls": 0,
  "evaluation_scope": "exploratory_validation_and_locked_historica